In [6]:
%pip install datasets math_verify vllm torch

(EngineCore pid=556) INFO 08-24 03:33:56 [core.py:1332] [shutdown] EngineCore: trigger received signal=SIGINT
(EngineCore pid=556) INFO 08-24 03:33:56 [core.py:1468] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=556) INFO 08-24 03:33:56 [core.py:1499] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=556) INFO 08-24 03:33:56 [core.py:1345] [shutdown] EngineCore: exiting busy loop
^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


INFO 08-24 03:34:00 [utils.py:612] [shutdown] Process manager: send sigterm to process EngineCore
WARNING 08-24 03:34:00 [core_client.py:725] [shutdown] MPClient: engine core exited unexpectedly; starting cleanup
INFO 08-24 03:34:00 [core_client.py:686] [shutdown] MPClient: start timeout=default
INFO 08-24 03:34:00 [core_client.py:688] [shutdown] MPClient: stopping engine manager
INFO 08-24 03:34:00 [core_client.py:690] [shutdown] MPClient: engine manager stopped
INFO 08-24 03:34:00 [core_client.py:691] [shutdown] MPClient: cleaning up background resources
INFO 08-24 03:34:00 [core_client.py:693] [shutdown] MPClient: complete


In [ ]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

data/train-00000-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00001-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

data/train-00002-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

data/train-00007-of-00010.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

data/train-00008-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00009-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93733 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [ ]:
import gc
import torch
from vllm import LLM, SamplingParams
from math_verify import parse, verify
import re

for var in ["llm", "generations"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    gpu_memory_utilization=0.6,
)
sampling_params = SamplingParams(
    n=8,
    max_tokens=2**10,
)

prompts = ["What is 13*17?"]
golds = ["221"]

generations = llm.generate(prompts, sampling_params)
for prompt, gold, generation in zip(prompts, golds, generations):
    gold = parse(gold)
    
    print("*"*50 + " Prompt " + "*"*50)
    print(prompt)
    
    for i, output in enumerate(generation.outputs):
        text = output.text
        answer = parse(text)
        correct = gold[0] == answer[0]
        
        print("*"*50 + f" Generation {i+1}: {answer[0]} ({"correct" if correct else "incorrect"}) ", "*"*50)
        if correct:
            print(text)

INFO 08-24 03:18:48 [api_utils.py:273] non-default args: {'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
INFO 08-24 03:18:48 [model.py:645] Resolved architecture: Qwen2ForCausalLM
WARNING 08-24 03:18:48 [model.py:2164] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 08-24 03:18:48 [model.py:2217] Casting torch.bfloat16 to torch.float16.
INFO 08-24 03:18:48 [model.py:1883] Using max model len 131072
INFO 08-24 03:18:48 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=556) INFO 08-24 03:19:05 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', skip_tokenizer_init=False, tokenizer_mod

[W824 03:19:06.421062159 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore pid=556) INFO 08-24 03:19:07 [model_runner.py:308] Loading model from scratch...
(EngineCore pid=556) ERROR 08-24 03:19:07 [fa_utils.py:273] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=556) INFO 08-24 03:19:10 [cuda.py:482] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=556) INFO 08-24 03:19:11 [weight_utils.py:867] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 3.31 GiB. Available RAM: 9.32 GiB.
(EngineCore pid=556) INFO 08-24 03:19:11 [weight_utils.py:890] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.20s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.20s/it]
(EngineCore pid=556) 


(EngineCore pid=556) INFO 08-24 03:19:14 [default_loader.py:430] Loading weights took 3.47 seconds
(EngineCore pid=556) INFO 08-24 03:19:15 [model_runner.py:329] Model loading took 3.45 GiB and 8.187015 seconds
(EngineCore pid=556) WARNING 08-24 03:19:15 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=556) INFO 08-24 03:19:17 [caching.py:335] reconstructed serializable fn from standalone compile artifacts. num_artifacts=3 num_submods=29
(EngineCore pid=556) INFO 08-24 03:19:17 [decorators.py:311] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/af68ece8784a1aa24e6a420cecb0d4c710dcfcb24120464d45f2e760c657287a/rank_0_0/model
(EngineCore pid=556) INFO 08-24 03:19:17 [monitor.py:53] torch.compile took 0.19 s in total
(EngineCore pid=556) INFO 08-24 03:19:17 [monitor.py:81] Initial profiling/warmup run took 0.09

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=556) WARNING 08-24 03:19:48 [jit_monitor.py:135] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 8/8 [00:19<00:00,  2.40s/it, est. speed input: 3.75 toks/s, output: 360.34 toks/s]

************************************************** Prompt **************************************************
What is 13*17?
************************************************** Generation 1: 3 (incorrect)  **************************************************
************************************************** Generation 2: 13 (incorrect)  **************************************************
************************************************** Generation 3: 221 (correct)  **************************************************
 To calculate 13 times 17, break down the factors to its easiest form.

First, consider 13 as 10 + 3. Then, 17 can be written as 10 + 7.

So, (10 + 3) x (10 + 7)

Next, multiply each part using the distributive property:

10 x 10 = 100
10 x 7 = 70
3 x 10 = 30
3 x 7 = 21

Add them up: 100 + 70 + 30 + 21 = 221

Alternatively, another approach is to add 10 + 7, which is 17, to a known product.

13 x 17 = 10 x 17 + 3 x 17 = 170 + 51 = 221

Both methods lead to the correct result by

In [ ]:
from datasets import Dataset

def generate_dataset(llm, sampling_params, prompts, golds):
    dataset_dict = {
        "prompt": [],
        "outputs": [],
        "advantages": [],
    }
    generations = llm.generate(prompts, sampling_params)
    for prompt, gold, generation in zip(prompts, golds, generations):
        outputs = []
        rewards = []
        for output in generation.outputs:
            outputs.append(output.text)
            rewards.append(1.0 if verify(parse(gold), parse(output.text)) else 0.0)
        rewards = torch.tensor(rewards)
        
        std = rewards.std()
        if std == 0:
            advantages = torch.zeros_like(rewards)
        else:
            advantages = (rewards - rewards.mean()) / std
        
        dataset_dict["prompt"].append(prompt)
        dataset_dict["outputs"].append(outputs)
        dataset_dict["advantages"].append(advantages)
    return Dataset.from_dict(dataset_dict)

sampling_params = SamplingParams(
    n=64,
    max_tokens=2**16,
)
input_ds = (
    ds
    .shuffle()
    .select(range(0, 1000))
)
output_ds = generate_dataset(
    llm=llm,
    sampling_params=sampling_params,
    prompts=input_ds["problem"],
    golds=input_ds["answers"]
)
output_ds.to_parquet("reg_grpo_1k.parquet")
output_ds